
# Customer Service RAG Bot — Student Version

This notebook builds a **Retrieval-Augmented Generation (RAG)** customer-support chatbot from a PDF.

### Learning goals
By the end of the notebook, students should understand how to:

1. Install the required Python libraries.
2. Load an API key safely.
3. Connect Google Drive when running in Google Colab.
4. Load a PDF into LangChain documents.
5. Split the PDF into smaller text chunks.
6. Convert chunks into embeddings.
7. Store and retrieve chunks with Chroma.
8. Combine retrieved context with an OpenAI chat model.
9. Test the RAG pipeline before building a user interface.
10. Launch a simple Gradio chat interface.

> **Run the notebook from top to bottom.**  
> The first run may take longer because Python packages and the embedding model must be downloaded.



## Cell 1 — Install the required packages

**What this cell does**

- `%pip install` installs packages into the current notebook environment.
- `langchain-*` packages provide the RAG building blocks.
- `langchain-chroma` provides the current Chroma integration.
- `sentence-transformers` provides the local multilingual embedding model.
- `pypdf` reads text from the PDF.
- `gradio` creates the web chat interface.
- `python-dotenv` allows local users to load an API key from a `.env` file.

This replaces the repeated installation cells in the original notebook and prevents a common error:
`ModuleNotFoundError: No module named 'langchain_chroma'`.


In [ ]:
# Install or update all libraries needed by this notebook.
# We install langchain-chroma explicitly because Chroma is now a separate integration package.
%pip install -q -U langchain langchain-openai langchain-community langchain-chroma langchain-huggingface langchain-text-splitters chromadb sentence-transformers gradio pypdf python-dotenv



## Cell 2 — Configuration, Google Drive, and the OpenAI API key

**What this cell does**

- Imports standard Python tools.
- Detects whether the notebook is running in Google Colab.
- Loads environment variables from a local `.env` file when available.
- Mounts Google Drive in Colab.
- Tries to read `OPENAI_API_KEY` from **Colab Secrets**.
- If the key is not available, asks for it securely with `getpass()`.
- Defines the PDF path and Chroma database folder in one place.

### Recommended API-key setup in Colab

1. Open the **Secrets** panel (key icon) in the left sidebar.
2. Create a secret named `OPENAI_API_KEY`.
3. Paste your OpenAI API key as its value.
4. Give the notebook access to the secret.

Do **not** write an API key directly into a notebook cell that you plan to share.


In [ ]:
# os is used for environment variables such as OPENAI_API_KEY.
import os

# Path gives us a clean, cross-platform way to work with file paths.
from pathlib import Path

# getpass asks for a secret without printing it on the screen.
from getpass import getpass

# load_dotenv reads variables from a local .env file when one exists.
from dotenv import load_dotenv

# Load a local .env file if present. This is useful outside Google Colab.
load_dotenv()

# Start by assuming that we are not running in Google Colab.
IN_COLAB = False

# Try to import Colab-specific tools.
try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    # If the import fails, the notebook is probably running locally or in Jupyter.
    pass

# In Colab, mount Google Drive so the PDF and Chroma database can persist.
if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)

# Try to read the API key from Colab Secrets first.
api_key = None

if IN_COLAB:
    try:
        api_key = userdata.get("OPENAI_API_KEY")
    except Exception:
        # The secret may not exist or the notebook may not have permission to use it.
        api_key = None
else:
    # Outside Colab, look for OPENAI_API_KEY in the environment or .env file.
    api_key = os.getenv("OPENAI_API_KEY")

# If no key was found, ask the user to enter one securely.
if not api_key:
    api_key = getpass("Enter your OpenAI API key: ").strip()

# Stop early with a clear message instead of failing later with a confusing API error.
if not api_key:
    raise ValueError(
        "OPENAI_API_KEY was not provided. Add it to Colab Secrets or a local .env file."
    )

# Save the key as an environment variable so LangChain/OpenAI can find it automatically.
os.environ["OPENAI_API_KEY"] = api_key

# Use Google Drive in Colab; otherwise use the folder containing the notebook/session.
BASE_DIR = Path("/content/drive/MyDrive") if IN_COLAB else Path.cwd()

# Name of the PDF used as the RAG knowledge source.
PDF_PATH = BASE_DIR / "customer_support_arabic_full.pdf"

# Folder where Chroma will store the vector database.
CHROMA_DIR = BASE_DIR / "chroma_customer_support"

# Set this to True when the PDF changes and you want to rebuild the vector database.
REBUILD_DB = False

# Embedding model used to convert text into numerical vectors.
EMBEDDING_MODEL = "intfloat/multilingual-e5-small"

# OpenAI model used to generate the final answer.
# You can change this value if your API account uses a different supported model.
OPENAI_MODEL = "gpt-4.1-mini"

# Print the important paths so students can verify them before continuing.
print(f"Running in Colab: {IN_COLAB}")
print(f"PDF path: {PDF_PATH}")
print(f"Chroma directory: {CHROMA_DIR}")



## Cell 3 — Verify the PDF before doing any expensive work

**Why this check matters**

The original notebook only printed a warning when the PDF was missing. The next cells then failed because they tried to use a file that did not exist.

This version raises a clear `FileNotFoundError` immediately and tells the student exactly what to fix.


In [ ]:
# Check that the PDF file exists at the path defined above.
if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"PDF not found: {PDF_PATH}\n"
        "Upload 'customer_support_arabic_full.pdf' to MyDrive in Colab, "
        "or change PDF_PATH to the correct file location."
    )

# Show the file size as a simple confirmation that we found the expected file.
pdf_size_mb = PDF_PATH.stat().st_size / (1024 * 1024)

print("PDF found successfully.")
print(f"PDF size: {pdf_size_mb:.2f} MB")



## Cell 4 — Load the PDF

**What happens here**

`PyPDFLoader` reads the PDF and creates one LangChain `Document` object per page.  
Each `Document` contains:

- `page_content`: the extracted text.
- `metadata`: information such as the page number and source file.

The original notebook had this important step commented out. That caused later cells to fail because `docs` and `chunks` were never created.


In [ ]:
# PyPDFLoader extracts text from a PDF and returns LangChain Document objects.
from langchain_community.document_loaders import PyPDFLoader

# Create a loader connected to our PDF.
loader = PyPDFLoader(str(PDF_PATH))

# Read every page of the PDF.
docs = loader.load()

# Stop if the loader returned no pages.
if not docs:
    raise ValueError("The PDF was opened, but no pages were loaded.")

# Count extracted characters to detect image-only/scanned PDFs.
total_characters = sum(len(doc.page_content.strip()) for doc in docs)

# If almost no text was extracted, the PDF may require OCR.
if total_characters < 100:
    raise ValueError(
        "Very little text was extracted from the PDF. "
        "The file may be scanned/image-based and may require OCR before using PyPDFLoader."
    )

print(f"Loaded {len(docs)} PDF pages.")
print(f"Extracted approximately {total_characters:,} text characters.")



## Cell 5 — Split the PDF into chunks

**Why chunking is needed**

An entire PDF is usually too large and too broad to retrieve as one object.  
The text splitter creates smaller overlapping pieces:

- `chunk_size=800`: target size of each chunk.
- `chunk_overlap=150`: repeats some text between neighboring chunks so ideas are not cut off abruptly.

These chunks become the searchable units in the vector database.


In [ ]:
# RecursiveCharacterTextSplitter breaks long text into smaller overlapping chunks.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configure the splitter.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      # Maximum approximate characters per chunk.
    chunk_overlap=150,   # Repeated characters between neighboring chunks.
    separators=["\n\n", "\n", "؟", ".", " ", ""],  # Prefer natural break points.
)

# Split all PDF pages into chunks while preserving each chunk's metadata.
chunks = splitter.split_documents(docs)

# Stop with a clear message if chunking unexpectedly produced nothing.
if not chunks:
    raise ValueError("No text chunks were created from the PDF.")

print(f"Created {len(chunks)} chunks.")
print("\nExample chunk:\n")
print(chunks[0].page_content[:700])



## Cell 6 — Create embeddings and build/load the Chroma vector database

**Key idea**

An embedding converts text into a vector of numbers. Texts with similar meanings get vectors that are closer together.

This notebook uses the multilingual `intfloat/multilingual-e5-small` model because the knowledge source can contain Arabic and English.

The logic is:

- If a saved Chroma database already exists, load it.
- Otherwise, embed the PDF chunks and build the database.
- If the source PDF changes, set `REBUILD_DB = True` in Cell 2.

`MMR` retrieval is used to balance **relevance** with **diversity** among the returned chunks.


In [ ]:
# shutil is only needed when we intentionally rebuild and delete the old vector database.
import shutil

# Chroma is the LangChain integration for the Chroma vector database.
from langchain_chroma import Chroma

# HuggingFaceEmbeddings runs a Sentence-Transformers embedding model locally.
from langchain_huggingface import HuggingFaceEmbeddings

# Create the multilingual embedding model.
# normalize_embeddings=True makes vector comparisons more stable for cosine-style similarity.
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={"normalize_embeddings": True},
)

# If the student explicitly requested a rebuild, remove the old database first.
if REBUILD_DB and CHROMA_DIR.exists():
    print("REBUILD_DB=True -> deleting the old Chroma database...")
    shutil.rmtree(CHROMA_DIR)

# A database counts as existing only if the folder exists and is not empty.
db_exists = CHROMA_DIR.exists() and any(CHROMA_DIR.iterdir())

if db_exists:
    # Load vectors that were created during an earlier run.
    print("Loading the existing Chroma database...")
    vectordb = Chroma(
        persist_directory=str(CHROMA_DIR),
        embedding_function=embeddings,
    )
else:
    # Build the database for the first time from the PDF chunks.
    print("Building the Chroma database for the first time...")
    vectordb = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=str(CHROMA_DIR),
    )

# Convert the vector database into a retriever.
# k=6 means the final retrieval returns up to six chunks.
# fetch_k=20 means MMR first considers a wider pool before selecting diverse results.
retriever = vectordb.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 6, "fetch_k": 20},
)

print("Retriever is ready.")



## Cell 7 — Test retrieval before adding the language model

This is an important debugging habit.

If retrieval is poor, the final chatbot will also be poor even if the language model is working perfectly.  
This cell tests only the **R** in RAG: **Retrieval**.

Change `test_query` to a question that you know is answered in the PDF.


In [ ]:
# Use a sample Arabic question to test semantic retrieval.
test_query = "ما هي سياسة الاستبدال؟"

# Ask the retriever for the most relevant chunks.
retrieved_docs = retriever.invoke(test_query)

# Show how many chunks were returned.
print(f"Retrieved {len(retrieved_docs)} chunks.\n")

# Print a short preview of each retrieved chunk.
for index, doc in enumerate(retrieved_docs, start=1):
    page = doc.metadata.get("page", "unknown")
    print(f"--- Result {index} | page={page} ---")
    print(doc.page_content[:500])
    print()



## Cell 8 — Create the RAG prompt and generation chain

The RAG chain has four conceptual steps:

1. The user's question is sent to the retriever.
2. Retrieved chunks are joined into one `context` string.
3. The prompt sends both the context and question to the chat model.
4. `StrOutputParser` converts the model response into a plain Python string.

The system prompt tells the assistant to:

- answer only from the retrieved context,
- admit when the answer is not present,
- suggest human support when needed,
- match Arabic or English,
- stay concise.


In [ ]:
# ChatOpenAI connects LangChain to an OpenAI chat model.
from langchain_openai import ChatOpenAI

# ChatPromptTemplate builds structured system + user messages.
from langchain_core.prompts import ChatPromptTemplate

# RunnablePassthrough sends the original user question forward unchanged.
from langchain_core.runnables import RunnablePassthrough

# StrOutputParser converts the AI message object into a simple text string.
from langchain_core.output_parsers import StrOutputParser


def format_docs(documents):
    """Join retrieved document chunks into one context string for the prompt."""
    return "\n\n".join(doc.page_content for doc in documents)


# Create the language model.
llm = ChatOpenAI(
    model=OPENAI_MODEL,
    temperature=0.2,  # Low temperature keeps support answers more consistent.
)

# The model must answer from the supplied context instead of inventing unsupported details.
system_prompt = """
أنت مساعد خدمة عملاء لشركة إلكترونيات.

التعليمات:
- استخدم فقط المعلومات الموجودة في "السياق" أدناه.
- إذا لم تجد الإجابة في السياق، قل بوضوح إن المعلومات غير متوفرة لديك.
- عند الحاجة، اقترح التواصل مع الدعم البشري.
- طابق لغة المستخدم: العربية أو الإنجليزية.
- اجعل الإجابة قصيرة وواضحة ومفيدة.

السياق:
{context}
"""

# Build a two-message chat prompt: system instructions + the user's question.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)

# Build the full Retrieval-Augmented Generation pipeline.
rag_chain = (
    {
        # Retrieve relevant chunks, then combine them into text.
        "context": retriever | format_docs,
        # Pass the user's original question through unchanged.
        "question": RunnablePassthrough(),
    }
    # Insert context and question into the prompt.
    | prompt
    # Ask the OpenAI model to generate the answer.
    | llm
    # Return plain text instead of an AIMessage object.
    | StrOutputParser()
)

print("RAG chain is ready.")



## Cell 9 — Test the complete RAG chain

Run one simple question before launching the interface.

If this cell succeeds, then:

- the API key works,
- the model name is available to your API account,
- retrieval works,
- prompt construction works,
- generation works.

This makes Gradio problems much easier to diagnose separately.


In [ ]:
# Ask one test question through the complete RAG pipeline.
test_answer = rag_chain.invoke("ما هي سياسة الاستبدال؟")

# Display the answer.
print(test_answer)



## Cell 10 — Build the Gradio chat interface

Gradio turns the working Python function into a simple web application.

Important details:

- The chatbot uses the modern **messages** format:
  `{"role": "user", "content": "..."}`.
- Empty messages are ignored.
- API/runtime errors are caught and shown as readable messages instead of crashing the interface.
- In Colab, `share=True` creates a temporary public link.
- Outside Colab, the app launches locally.

The original notebook stored messages as dictionaries but did not explicitly configure the chatbot's message format. Making the format explicit improves compatibility with current Gradio versions.


In [ ]:
# Import Gradio, which provides the web interface components.
import gradio as gr


def respond(message, chat_history):
    """
    Send one user message through the RAG chain and update the chat history.

    Parameters
    ----------
    message : str
        The newest user message.
    chat_history : list
        Previous messages in Gradio's messages format.

    Returns
    -------
    tuple
        Updated chat history and an empty string to clear the input box.
    """

    # Always work with a list, even if the chat has just started.
    chat_history = list(chat_history or [])

    # Ignore blank messages.
    if not message or not message.strip():
        return chat_history, ""

    # Remove unnecessary spaces around the user's question.
    clean_message = message.strip()

    try:
        # Run the complete RAG pipeline.
        bot_reply = rag_chain.invoke(clean_message)
    except Exception as error:
        # Show a readable error in the chat rather than crashing the entire interface.
        bot_reply = (
            "An error occurred while generating the answer.\n\n"
            f"Technical detail: {error}"
        )

    # Add the user's message using Gradio's messages format.
    chat_history.append(
        {"role": "user", "content": clean_message}
    )

    # Add the assistant's answer.
    chat_history.append(
        {"role": "assistant", "content": bot_reply}
    )

    # Return the updated conversation and clear the textbox.
    return chat_history, ""


# Create the web interface.
with gr.Blocks() as demo:
    # Add a title and short instruction.
    gr.Markdown(
        "## 🤖 Smart Customer Support — RAG Bot\n"
        "Ask a question in Arabic or English about the information in the PDF."
    )

    # Explicitly use the modern role/content message format.
    chatbot = gr.Chatbot(
        label="Customer Support AI Assistant",
        type="messages",
        height=450,
    )

    # Text box where the user types a question.
    msg = gr.Textbox(
        placeholder="اكتب استفسارك هنا... / Type your question here...",
        label="Your message / رسالتك",
    )

    # Button that clears the conversation.
    clear = gr.Button("Clear chat / مسح الدردشة")

    # Pressing Enter calls respond(message, chat_history).
    msg.submit(
        fn=respond,
        inputs=[msg, chatbot],
        outputs=[chatbot, msg],
    )

    # Reset both the chatbot and the input box.
    clear.click(
        fn=lambda: ([], ""),
        inputs=None,
        outputs=[chatbot, msg],
    )

# In Colab, create a temporary share link.
# In local Jupyter, use the local URL instead.
demo.launch(share=IN_COLAB)



## Troubleshooting guide

### 1. `PDF not found`
Check `PDF_PATH` in Cell 2. In Colab, the default notebook expects:

`/content/drive/MyDrive/customer_support_arabic_full.pdf`

### 2. The PDF loads but almost no text is extracted
The PDF may be scanned images. `PyPDFLoader` does not perform OCR. Convert/OCR the PDF first, then rerun the notebook.

### 3. `ModuleNotFoundError`
Rerun **Cell 1**, then rerun the notebook from Cell 2 downward. If Colab reports that a runtime restart is required after package upgrades, restart the runtime once and run all cells again.

### 4. OpenAI authentication error
Make sure the Colab secret is named exactly:

`OPENAI_API_KEY`

and that notebook access is enabled.

### 5. Model not found / access error
Change `OPENAI_MODEL` in Cell 2 to a text/chat model available to your OpenAI API project.

### 6. The bot answers from old PDF content
Set:

`REBUILD_DB = True`

run the vector-database cell once, then set it back to `False`.

### 7. Retrieval results are irrelevant
Check Cell 7 before debugging the LLM. Try:

- asking a question clearly answered in the PDF,
- adjusting `chunk_size`,
- increasing/decreasing `k`,
- rebuilding the database if the PDF changed.

---

### RAG flow to remember

**PDF → Documents → Chunks → Embeddings → Chroma → Retriever → Prompt + Context → LLM → Answer**
